In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_classic.schema import Document
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import StrOutputParser 

In [3]:
text_loader = TextLoader("./langchain_sample.txt")
documents = text_loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, 
    chunk_overlap = 50, 
    separators = ["\n\n", "\n", " ", ""], 
    length_function = len
    )
texts = splitter.split_documents(documents)
texts

[Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a tradit

In [5]:
from langchain_community.retrievers import BM25Retriever

retriever_sparse = BM25Retriever.from_documents(texts, k=20)

In [6]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.vectorstores import FAISS

In [8]:
enbeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(texts, enbeddings)

retriever_dense = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":20})

In [9]:
retriever_sparse, retriever_dense

(BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001F2402642D0>, k=20),
 VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F2422BAD10>, search_kwargs={'k': 20}))

In [10]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers = [retriever_sparse, retriever_dense],
    weights = [0.3, 0.7]
    )

hybrid_retriever

EnsembleRetriever(retrievers=[BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001F2402642D0>, k=20), VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F2422BAD10>, search_kwargs={'k': 20})], weights=[0.3, 0.7])

In [11]:
retriever_sparse.invoke("What is LangChain?")

[Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes, which makes it scalable for large document stores.\nAgents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='Dense retrieval uses embeddings to match query and document

In [12]:
retriever_dense.invoke("What is LangChain?")

[Document(id='f801b3c0-e64a-4702-87bd-bc0075a0a8ac', metadata={'source': './langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='896ae471-5614-4ca3-a438-16177c0d29d9', metadata={'source': './langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='e38752bd-5472-4893-8439-cf8b6dd7035c', metadata={'source': './langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hug

In [13]:
hybrid_retriever.invoke("What is LangChain?")

[Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use case

In [14]:
query = "What is LangChain?"

In [15]:
# Retrieve separately
dense_docs = retriever_dense.invoke(query)
sparse_docs = retriever_sparse.invoke(query)

# Assign ranks
def assign_ranks(docs):
    return {id(doc): rank for rank, doc in enumerate(docs, start=1)}

dense_ranks = assign_ranks(dense_docs)
sparse_ranks = assign_ranks(sparse_docs)

# Union documents
all_docs = {id(doc): doc for doc in dense_docs + sparse_docs}

def reciprocal_rank_fusion(dense_ranks, sparse_ranks, k=60):
    scores = {}

    for doc_id in set(dense_ranks) | set(sparse_ranks):
        score = 0.0
        if doc_id in dense_ranks:
            score += 1 / (k + dense_ranks[doc_id])
        if doc_id in sparse_ranks:
            score += 1 / (k + sparse_ranks[doc_id])
        scores[doc_id] = score

    return scores
# Calculate RRF scores
rrf_scores = reciprocal_rank_fusion(dense_ranks, sparse_ranks, k=60)
# Rank documents based on RRF scores
ranked_docs = sorted(all_docs.values(), key=lambda doc: rrf_scores[id(doc)], reverse=True)
ranked_docs


[Document(id='f801b3c0-e64a-4702-87bd-bc0075a0a8ac', metadata={'source': './langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='896ae471-5614-4ca3-a438-16177c0d29d9', metadata={'source': './langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with ex

In [16]:
def apply_rule_boost(scores, docs, length_boost=0.05, max_tokens=300):
    for doc_id, doc in docs.items():
        if len(doc.page_content.split()) <= max_tokens:
            scores[doc_id] += length_boost
    return scores

# Compute fused scores
rrf_scores = reciprocal_rank_fusion(dense_ranks, sparse_ranks, k=60)

# Apply optional boosts
rrf_scores = apply_rule_boost(rrf_scores, all_docs)

# Sort documents
reranked_docs = sorted(
    all_docs.values(),
    key=lambda d: rrf_scores[id(d)],
    reverse=True
)

# Take top-K for LLM
final_docs = reranked_docs[:8]
final_docs

[Document(id='f801b3c0-e64a-4702-87bd-bc0075a0a8ac', metadata={'source': './langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': './langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='896ae471-5614-4ca3-a438-16177c0d29d9', metadata={'source': './langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with ex

In [55]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document

from typing import List
from pydantic import Field

class ExplicitHybridRetriever(BaseRetriever):
    dense: BaseRetriever = Field(...)
    sparse: BaseRetriever = Field(...)
    k: int = 60
    top_k: int = 8

    def _get_relevant_documents(self, query: str) -> List[Document]:
        dense_docs = self.dense.invoke(query)
        sparse_docs = self.sparse.invoke(query)

        dense_ranks = {id(d): i + 1 for i, d in enumerate(dense_docs)}
        sparse_ranks = {id(d): i + 1 for i, d in enumerate(sparse_docs)}

        all_docs = {id(d): d for d in dense_docs + sparse_docs}

        scores = {}
        for doc_id in all_docs:
            scores[doc_id] = (
                (1 / (self.k + dense_ranks[doc_id]) if doc_id in dense_ranks else 0)
              + (1 / (self.k + sparse_ranks[doc_id]) if doc_id in sparse_ranks else 0)
            )

        ranked_docs = sorted(
            all_docs.values(),
            key=lambda d: scores[id(d)],
            reverse=True
        )

        return ranked_docs[:self.top_k]




explicit_hybrid_retriever = ExplicitHybridRetriever(
    dense = retriever_dense,
    sparse = retriever_sparse,
    top_k=10
)

In [ ]:
chat_model = init_chat_model("gpt-3.5-turbo", temperature=0)

prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{input}"

Documents:
{context}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format:
Return a JSON array of all document indices in ranked order.
Example: [2,1,3,0]
""")

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain


stuffed_docs_chain = create_stuff_documents_chain(
    llm=chat_model,
    prompt=prompt,
    output_parser=StrOutputParser()
    )

retrival_chain = create_retrieval_chain(
    retriever=explicit_hybrid_retriever,
    combine_docs_chain=stuffed_docs_chain
    )

answer = retrival_chain.invoke({"input": query})
answer['answer']